# 02 — External validation on MIMIC-III

**Requirements:** PhysioNet credentialed access to MIMIC-III v1.4 linked to BigQuery, and
the exported development models (`run_development.py --stages export`).

Never commit `data/mimic_cache/` or patient-level MIMIC outputs (PhysioNet DUA).
Check small cells (< 10 patients) in subgroup tables before publication.

In [ ]:
# Colab / local setup. Results are written under HF_PROJECT_ROOT
# (set it to a Drive folder on Colab so outputs persist).
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_PROJECT_ROOT'] = '/content/drive/MyDrive/HF_Revisit_Project'
    if not os.path.exists('/content/HF-Survival-Analysis-Leakage-Revisit'):
        subprocess.run(['git', 'clone', '-b', 'claude/ecstatic-wright-61dz0b', 'https://github.com/elifnuroksuzz/HF-Survival-Analysis-Leakage-Revisit.git', '/content/HF-Survival-Analysis-Leakage-Revisit'], check=True)
    os.chdir('/content/HF-Survival-Analysis-Leakage-Revisit')
    subprocess.run(['git', 'pull', '-q', 'origin', 'claude/ecstatic-wright-61dz0b'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[mimic]'], check=True)
sys.path.insert(0, 'src')
from hfrevisit import config as C
P = C.get_paths()
print('Project root:', P.root)
if IN_COLAB:
    from google.colab import auth; auth.authenticate_user()

In [ ]:
# Step 1: train and save the development models on the Kaggle data (~1-2 min)
!python scripts/run_development.py --download --stages export

In [ ]:
# Step 2: external validation (BigQuery; ~10-30 min)
GCP_PROJECT = 'physionet-data-491418'   # <- your billing project with PhysioNet access
!python scripts/run_external_validation.py --gcp-project {GCP_PROJECT} --horizon 90

In [ ]:
# Step 3: draw 200 echo reports for manual EF annotation (fill the sheet, then run 'score')
!python scripts/ef_annotation.py sample
# !python scripts/ef_annotation.py score

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_columns', 40)
def table(name): return pd.read_csv(P.tables / f'{name}.csv')
def figure(name): display(Image(filename=str(P.figures / f'{name}.png'), width=900))

In [ ]:
table('mimic_primary_attrition')

In [ ]:
table('mimic_primary_performance').round(3)

In [ ]:
for s in ['sens_numeric_ef', 'sens_complete_cpk', 'sens_age40']:
    print(s); display(table(f'mimic_{s}_performance').round(3))

In [ ]:
figure('fig5_mimic_calibration')
table('mimic_primary_recalibration').round(3)

In [ ]:
display(table('mimic_primary_subgroups').round(3)); table('mimic_primary_subgroup_gaps').round(3)